##### Create a baseline model to establish a simple, interpretable benchmark for accident risk prediction. This benchmark is used to evaluate whether more complex ML models provide meaningful improvement.

In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_parquet("modelling_dataset.parquet", engine="fastparquet")
df.head(20)


In [ ]:
# Convert accident counts into binary (1: accident(s) occured, 0: no accidents occured)
df["accident_binary"] = (df["Accident_Count"] > 0).astype(int)

## Split Data

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# time based split
split_80 = df["Time_Stamp"].quantile(0.80)

# rows that come before 80th percentile with respect to the time stamp are assigned to training; the rest to testing
train_df = df[df["Time_Stamp"] < split_80].copy()
test_df  = df[df["Time_Stamp"] >= split_80].copy()

In [ ]:
# calculate accident probability per Zone_ID, Hour 
# Done by taking the mean of the binary accident indicator within each group
baseline = train_df.groupby(["Zone_ID", "Hour"])["accident_binary"].mean()

In [ ]:
# Apply baseline probabilities to the test set
test_df = test_df.merge(
    baseline.rename("baseline_prob"),
    on=["Zone_ID", "Hour"],
    how="left"
)

## Fill missing values

In [ ]:
# fill missing probabilities for unseen zone, hour combos with overall accident rate in training set
global_mean = train_df["accident_binary"].mean()
test_df["baseline_prob"] = test_df["baseline_prob"].fillna(global_mean)

## Baseline Model Evaluation

In [ ]:
roc_auc = roc_auc_score(test_df["accident_binary"], test_df["baseline_prob"])
avg_p = average_precision_score(test_df["accident_binary"],test_df["baseline_prob"])
print("Baseline ROC-AUC:", roc_auc)
print("AP:", avg_p)

In [ ]:
# convert probabilities to binary predictions using a threshold
# a lower threshold is used to better identify rare accident events
threshold = 0.2
test_df["baseline_pred"] = (test_df["baseline_prob"] >= threshold).astype(int)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_true = test_df["accident_binary"]
y_pred = test_df["baseline_pred"]

print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))